# FXS / 08 — Discovery on the other 12 pairs, 2021–2025

Iván (2026-11-11): FXS 06 found structure in EURUSD, USDJPY and EURJPY on 2021–2025 that was not there (or was weaker) in 2016–2020. Run the same scan on the **12 pairs left out**: AUDJPY, AUDNZD, AUDUSD, CADCHF, CADJPY, EURCHF, EURGBP, GBPJPY, GBPUSD, NZDUSD, USDCAD, USDCHF — over three windows:

| Window | Question |
|---|---|
| 2016–2020 | the original discovery (should reproduce FXS 01) |
| **2021–2025** | do the original structures still pass discovery on later data? are there **new** structures? |
| 2016–2025 | everything pooled |

Same method as FXS 01/002: both clocks (New York 17:00→17:00, UTC), every interval starting every 5 minutes and 15–240 minutes long (2,704 tests per pair and window), D1 Benjamini–Hochberg within the pair (q ≤ 0.05), D2 same sign in ≥ 80% of the years (4 of 5, 8 of 10), D3 |mean| > median spread + 0.70 bp commission, de-duplication, ≤ 10 candidates, rotated-day placebos (D0).

**What can confirm a 2021–2025 find.** 2016–2020 is a backward check only (it was scanned in FXS 01 for these pairs). FXS 07 showed the 06 finds were mostly negative in 2020 and ≈ 0 in 2026, so a 2021–2025 find is a hypothesis, nothing more. **2026 is not read here.** The 2021–2025 candidates are frozen in `08/08_candidates_frozen.csv` (section 6) so that a separate, pre-registered one-look test on 2026 can follow.

The scan saves each (window, pair) result in `08/_scan/` as it finishes, so a stopped run resumes where it left off.

In [ ]:
import json, os, sys, tempfile, math
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PAIRS = ["AUDJPY", "AUDNZD", "AUDUSD", "CADCHF", "CADJPY", "EURCHF", "EURGBP", "GBPJPY", "GBPUSD", "NZDUSD", "USDCAD", "USDCHF"]
if os.environ.get("FXS_PAIRS"):
    PAIRS = os.environ["FXS_PAIRS"].split(",")
COMMISSION_PER_LOT = 3.0
SPREAD_FACTOR = 0.9
WINDOWS = {"2016-2020": (dt.date(2016, 1, 1), dt.date(2020, 12, 31), "discovery"),
           "2021-2025": (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"),
           "2016-2025": (dt.date(2016, 1, 1), dt.date(2025, 12, 31), "confirmation")}
N_PLACEBO = 20

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs08_")
    os.environ.pop("DUKASCOPY_DIR", None)
    N_PLACEBO = 3
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg0 = F.FXSConfig(n_placebo=N_PLACEBO)
BRANCH = Path.cwd().parent
OUT = Path.cwd() / ("08_synthetic" if SYNTHETIC else "08"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 200)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(PAIRS):
        F.write_synthetic_candles(base, p, dt.date(2015, 12, 28), dt.date(2025, 12, 31), seed=51 + i, drift_bps=-3.0 if i == 0 else 0.0,
                                  drift_ny=(4, 8), price=150.0 if p.endswith("JPY") else 1.10)
    orig = pd.DataFrame([{"id": f"{PAIRS[0]}-01", "pair": PAIRS[0], "clock": "NY", "start": 660, "end": 900, "direction": -1}])
else:
    orig = F.load_frozen_candidates(BRANCH).query("pair in @PAIRS").reset_index(drop=True)
orig["rule"] = [f"{'buy' if d > 0 else 'sell'} {F.slot_label(c, s)} -> {F.slot_label(c, e).split()[1]}" for c, s, e, d in
                zip(orig["clock"], orig["start"], orig["end"], orig["direction"])]
print("FXS 01 candidates of these pairs (all failed 2021-2025 confirmation):")
display(orig[["id", "pair", "rule"]])

## 1. Scan every pair in every window

12 pairs × 3 windows × (scan + 20 placebos). The first run builds minute caches for the new windows; expect several hours. Interrupt and rerun freely: finished (window, pair) results are reused.

In [ ]:
import pickle
CACHE = OUT / "_scan"; CACHE.mkdir(exist_ok=True)
scans, cands, d0 = {}, {}, []
for w, (a, b, stage) in WINDOWS.items():
    n_years = b.year - a.year + 1
    cfg = replace(cfg0, min_years_same_sign=math.ceil(0.8 * n_years))
    for p in PAIRS:
        ck = CACHE / f"{w}_{p}.pkl"
        if ck.exists():
            sc, cd, real, p95 = pickle.loads(ck.read_bytes())
        else:
            panels = F.load_panels([p], a, b, stage, cfg)
            sc = pd.concat([F.scan_panel(panels[(p, c)], cfg) for c in F.CLOCKS], ignore_index=True)
            sc, cd = F.select_candidates(sc, cfg)
            cd["pair"] = p
            if len(cd):
                cd["id"] = [f"new[{w}] {i}" for i in cd["id"]]      # not to be confused with the FXS 01 ids
            sc["D123"] = sc["D1"] & sc["D2"] & sc["D3"]
            pl = F.placebo_counts(panels, cfg)
            real = int(sc["D123"].sum()); p95 = float(np.percentile(pl["D1_D2_D3"], 95))
            ck.write_bytes(pickle.dumps((sc, cd, real, p95)))
            del panels
        scans[(w, p)], cands[(w, p)] = sc, cd
        d0.append({"window": w, "pair": p, "years needed (D2)": cfg.min_years_same_sign, "D1": int(sc["D1"].sum()),
                   "D1+D2": int((sc["D1"] & sc["D2"]).sum()), "D1+D2+D3": real, "placebo p95": p95,
                   "D0": "STRUCTURE_FOUND" if real > p95 else "NO_STRUCTURE", "candidates": len(cd)})
        print(f"{w} {p}: D1 {int(sc['D1'].sum())}, D1-D3 {real}, placebo p95 {p95:.1f}, candidates {len(cd)}")
d0 = pd.DataFrame(d0)
display(d0.pivot(index="pair", columns="window", values=["D1+D2+D3", "D0", "candidates"]))
d0.to_csv(OUT / "08_summary_by_window.csv", index=False)

## 2. The original structures in each window

The exact frozen interval looked up in each window's scan: mean (bps at the mid, in its trade direction), t, BH q, years with the same sign, filters passed, and whether it (or an overlapping interval) was selected as a candidate. `rank` = position by |t| among the pair's 2,704 tests.

In [ ]:
rows = []
for _, r in orig.iterrows():
    seg = F._utc_segments(r["clock"], int(r["start"]), int(r["end"]))
    for w in WINDOWS:
        sc = scans[(w, r["pair"])]
        sc = sc.assign(rank=sc["t"].abs().rank(ascending=False, method="min").astype(int))
        hit = sc[(sc["clock"] == r["clock"]) & (sc["start"] == r["start"]) & (sc["end"] == r["end"])]
        if hit.empty:
            continue
        h = hit.iloc[0]
        cd = cands[(w, r["pair"])]
        over = [c["id"] for _, c in cd.iterrows() if F._overlap(seg, F._utc_segments(c["clock"], int(c["start"]), int(c["end"])))]
        rows.append({"structure": r["id"], "rule": r["rule"], "window": w, "mean bps (trade dir.)": h["mean_bps"] * r["direction"],
                     "t (trade dir.)": h["t"] * r["direction"], "q": h["q"], "years same sign": h["years_same_sign"],
                     "D1": h["D1"], "D2": h["D2"], "D3": h["D3"], "rank": int(h["rank"]),
                     "selected": bool(((cd["clock"] == r["clock"]) & (cd["start"] == r["start"]) & (cd["end"] == r["end"])).any()) if len(cd) else False,
                     "overlapping candidate": ", ".join(over) or "-"})
status = pd.DataFrame(rows)
display(status.round(3)); status.to_csv(OUT / "08_original_structures.csv", index=False)

## 3. Candidates found in each window

`new?` = the candidate does not overlap any original structure of the same pair (in UTC time, New York intervals under both daylight-saving offsets).

In [ ]:
allc = []
for (w, p), cd in cands.items():
    if cd.empty:
        continue
    o = orig[orig["pair"] == p]
    for _, c in cd.iterrows():
        seg = F._utc_segments(c["clock"], int(c["start"]), int(c["end"]))
        match = [x["id"] for _, x in o.iterrows() if F._overlap(seg, F._utc_segments(x["clock"], int(x["start"]), int(x["end"])))]
        allc.append({"window": w, "pair": p, "candidate": c["id"], "trade": f"{'buy' if c['direction'] > 0 else 'sell'} {c['start_time']} -> {c['end_time'].split()[1]}",
                     "minutes": c["minutes"], "mean bps": abs(c["mean_bps"]), "t": abs(c["t"]), "q": c["q"], "years same sign": c["years_same_sign"],
                     "cost bps": c["cost_bps"], "pooled Bonferroni |t|>=4.84": abs(c["t"]) >= 4.84,
                     "overlaps original": ", ".join(match) or "-", "new?": not match})
allc = pd.DataFrame(allc)
for w in WINDOWS:
    print(f"\n=== {w} ===")
    display(allc[allc["window"] == w].drop(columns="window").round(3))
allc.to_csv(OUT / "08_candidates.csv", index=False)

## 4. The stacked day: 2016–2020 vs 2021–2025

Mean cumulative mid return from the day start (bps) with ± 2 SE bands, per pair, New York clock. Shaded: FXS 01 candidates (green = long, red = short).

In [ ]:
ncol = 3
fig, axes = plt.subplots(math.ceil(len(PAIRS) / ncol), ncol, figsize=(19, 3.6 * math.ceil(len(PAIRS) / ncol)), squeeze=False)
for i, p in enumerate(PAIRS):
    for clock in ("NY",):
        ax = axes[i // ncol, i % ncol]
        for w, col in (("2016-2020", "C0"), ("2021-2025", "C3")):
            a, b, stage = WINDOWS[w]
            pr = F.profile(F.load_panels([p], a, b, stage, cfg0, clocks=(clock,))[(p, clock)])
            x = pr.index / 60
            ax.plot(x, pr["cum_mean_bps"], color=col, lw=1.1, label=w)
            ax.fill_between(x, pr["cum_mean_bps"] - 2 * pr["cum_se_bps"], pr["cum_mean_bps"] + 2 * pr["cum_se_bps"], color=col, alpha=0.12)
        for _, r in orig[(orig["pair"] == p) & (orig["clock"] == clock)].iterrows():
            ax.axvspan(r["start"] / 60, r["end"] / 60, color="green" if r["direction"] > 0 else "red", alpha=0.12)
        ax.axhline(0, color="black", lw=0.5)
        ax.set_xticks(range(0, 25, 2)); ax.set_xticklabels([F.slot_label(clock, 60 * h).split()[1] for h in range(0, 25, 2)], rotation=45, fontsize=8)
        ax.set_title(f"{p}, NY clock (bps)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "08_stacked_day_by_period.png", dpi=110); plt.show()

## 5. 15-minute blocks: which times of day changed?

t-statistic of each 15-minute block, 2016–2020 vs 2021–2025 (New York clock). Blocks with |t| ≥ 3 in one period and the opposite sign or |t| < 1 in the other are listed.

In [ ]:
chg = []
for p in PAIRS:
    bc = {}
    for w in ("2016-2020", "2021-2025"):
        a, b, stage = WINDOWS[w]
        bc[w] = F.block_consistency(F.load_panels([p], a, b, stage, cfg0, clocks=("NY",))[(p, "NY")], 15).set_index("time")
    j = bc["2016-2020"][["mean_bps", "t"]].join(bc["2021-2025"][["mean_bps", "t"]], lsuffix=" 16-20", rsuffix=" 21-25")
    j["pair"] = p
    flag = ((j["t 16-20"].abs() >= 3) & ((np.sign(j["t 16-20"]) != np.sign(j["t 21-25"])) | (j["t 21-25"].abs() < 1))) | \
           ((j["t 21-25"].abs() >= 3) & ((np.sign(j["t 21-25"]) != np.sign(j["t 16-20"])) | (j["t 16-20"].abs() < 1)))
    chg.append(j[flag].reset_index())
chg = pd.concat(chg, ignore_index=True)
display(chg.round(2)); chg.to_csv(OUT / "08_changed_blocks.csv", index=False)

## 6. The 2021–2025 candidates in pips, and the frozen file

Each 2021–2025 candidate traded every day at the Dukascopy bid/ask × `SPREAD_FACTOR` plus `COMMISSION_PER_LOT` per lot round trip: net pips per trade by year, 2016–2025. **2021–2025 is in sample** (the candidates were picked there); 2016–2020 is the only out-of-sample number in this notebook. The candidates are then written to `08/08_candidates_frozen.csv` with a SHA-256 in `summary.json`; that file is what a 2026 one-look test would use, unchanged.

In [ ]:
new = pd.concat([cd for (w, p), cd in cands.items() if w == "2021-2025" and len(cd)], ignore_index=True) if len(allc) else pd.DataFrame()
pip_tab, frozen_sha = pd.DataFrame(), None
if len(new):
    new = new.reset_index(drop=True)
    new["id"] = [f"{p}-N{k + 1}" for p, k in zip(new["pair"], new.groupby("pair").cumcount())]
    parts = [F.structure_detail(new, a, b, st, cfg0, with_price=True, spread_factor=SPREAD_FACTOR) for a, b, st in
             (WINDOWS["2016-2020"], WINDOWS["2021-2025"])]
    G = pd.concat([x[0] for x in parts]).sort_index().fillna(0.0)
    P = pd.concat([x[2] for x in parts]).sort_index().ffill().bfill()
    PAIR_OF = new.set_index("id")["pair"]
    pips = pd.DataFrame({i: F.bps_to_pips(G[i] - F.commission_bps_per_lot(PAIR_OF[i], COMMISSION_PER_LOT), P[i], PAIR_OF[i])
                         for i in G.columns}, index=G.index).where(G != 0)
    yr = pips.groupby(pips.index.year).mean().T
    rows = []
    for i, r in new.set_index("id").iterrows():
        row = {"id": i, "trade": f"{'buy' if r['direction'] > 0 else 'sell'} {r['pair']} {r['start_time']} -> {r['end_time'].split()[1]}", "minutes": r["minutes"]}
        for per, (a, b) in {"2016-2020": (2016, 2020), "2021-2025": (2021, 2025)}.items():
            x = pips[i][(pips.index.year >= a) & (pips.index.year <= b)].dropna()
            row[f"{per} net pips"] = x.mean(); row[f"{per} t"] = x.mean() / (x.std() / np.sqrt(len(x)))
        rows.append(row)
    pip_tab = pd.DataFrame(rows).set_index("id").join(yr.round(2))
    display(pip_tab.round(2)); pip_tab.to_csv(OUT / "08_new_candidates_pips.csv")
    ff = OUT / "08_candidates_frozen.csv"
    new.drop(columns=[c for c in new.columns if c not in ("id", "pair", "clock", "start_time", "end_time", "minutes", "direction", "mean_bps", "t", "q",
                                                         "years_same_sign", "cost_bps", "start", "end")]).to_csv(ff, index=False, lineterminator="\n")
    import hashlib
    frozen_sha = hashlib.sha256(ff.read_bytes()).hexdigest()
    print("frozen", ff, frozen_sha)
else:
    print("no 2021-2025 candidates in these pairs")

In [ ]:
summary = {"study": "FXS08", "descriptive": True, "synthetic": SYNTHETIC, "pairs": PAIRS, "windows": list(WINDOWS),
           "by_window": d0.to_dict(orient="records"), "original_structures": status.round(4).to_dict(orient="records"),
           "candidates": allc.round(4).to_dict(orient="records") if len(allc) else [],
           "new_candidates_pips": pip_tab.round(4).reset_index().to_dict(orient="records") if len(pip_tab) else [],
           "candidates_sha256": frozen_sha, "commission_usd_per_lot_rt": COMMISSION_PER_LOT, "spread_factor": SPREAD_FACTOR,
           "note": "2026 not read. 2021-2025 candidates are in sample; 2016-2020 is a backward check; a 2026 one-look test needs a pre-registration first."}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")